# Vicuña · Plantilla de experimento (T20 · HU-09)

1. Sube este notebook a **tu** Kaggle como **privado**, añade los datos de la competencia y elige **GPU T4**.
2. Edita **solo la celda «Tu experimento»** (y la de «Código propio» si cambias la red o las entradas). Un solo cambio por candidato.
3. Ejecuta todo (`Save & Run All`). Con 1 candidato tarda unos 20 min en T4 (la base se vuelve a correr).
4. Copia la fila que imprime la última celda a `docs/modelo/experimentos.md` y abre tu PR. Otra persona la verifica.

**Protocolo:** cortes A (16–30 jun) y B (1–15 jun, entrenando solo con lo anterior), semillas 0 y 1, época final fija. Se **acepta** si la media sube ≥ 0.001 **y** mejora en A y en B. Se registra también lo que no funciona. No se envía a Kaggle desde aquí.

Entrena solo con ventanas completas (`train/ventanas.py`, T11), así que los números de la base no son idénticos a los de la ronda 2: compara contra la base de **esta** corrida.

In [ ]:
# ==== Tu experimento: edita solo esta celda ====
ID = 'E2'                       # ID en el plan: E1, E2, ...
AUTOR = 'tu_usuario'
HIPOTESIS = 'Una tasa de aprendizaje menor mejora'
CANDIDATOS = {                  # nombre -> UN solo cambio sobre BASE
    'lr1e-3': dict(lr=1e-3),
}
QUICK = False                   # True: prueba corta para ver que todo corre (los scores no valen)

# Base vigente: ronda 2 (residuo crudo, decodificador profundo, dropout 0.1)
BASE = {'w': 64, 'bloques': [1, 2, 4, 1], 'enc_prof': False, 'dropout': 0.1, 'wd': 0.0001, 'lags': 3, 'skip': False,
        'residuo': 'crudo', 'terreno': False, 'profundo': True, 'peso_mj': 1, 'epocas': 25,
        'lr': 2e-3, 'lote': 32, 'entradas': 'gfs', 'arq': 'base'}

In [ ]:
import glob, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

def find(name):
    # Busca en Kaggle, en la carpeta data/ del repo (corriendo desde train/ o desde la raíz) y en la carpeta actual
    hits = (glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'../data/**/{name}', recursive=True)
            + glob.glob(f'data/**/{name}', recursive=True) + glob.glob(f'**/{name}', recursive=True))
    assert hits, f'No encuentro {name}'
    return hits[0]

tr = np.load(find('train.npz'))
te = np.load(find('test.npz'))
st = np.load(find('static.npz'))

Xtr, Ytr, ttr = tr['X'], tr['Y'], tr['times'].astype(str)
Xte, tte = te['X'], te['times'].astype(str)
dem = st['dem']
VARS = ['RH', 'T', 'U', 'V']; UNITS = ['%', 'K', 'm/s', 'm/s']

print('Train  X (GFS):', Xtr.shape, ' Y (WRF):', Ytr.shape, ' horas:', ttr[0], '→', ttr[-1])
print('Test   X (GFS):', Xte.shape, ' horas:', tte[0], '→', tte[-1])
print('DEM:', dem.shape, f'{dem.min():.0f}–{dem.max():.0f} m')

In [ ]:
# COPIA EXACTA de train/validacion.py: no editar aquí
"""
Validación temporal con la métrica del concurso (T02 · HU-09). Solo NumPy.

Corte temporal fijo: validación = últimas dos semanas de junio (desde 2024-06-16 00 UTC);
entrenamiento = todo lo anterior. Nunca se hace split aleatorio.

Hay dos scores y no son comparables entre sí:
  score_local    el del starter: cada RMSE se divide entre el RMSE de la climatología de train
                 medido en la propia validación (climatología = 0). Todas las horas.
  score_oficial  fórmula del leaderboard, con sus constantes fijas, sobre las horas que Kaggle
                 evalúa (00, 03, ..., 21 UTC). Es el que se usa para comparar experimentos.

El notebook lleva una copia exacta de este archivo en una celda para correr solo en Kaggle;
tests/test_validacion.py comprueba que ambas sean iguales.
"""
import numpy as np

VARS = ("RH", "T", "U", "V")
INICIO_VAL = "2024061600"
ESCALAS_OFICIALES = np.array([25.360, 2.0652, 2.6776, 2.0876])
HORAS_EVALUADAS = tuple(range(0, 24, 3))


def horas(times):
    """Hora UTC (0-23) de cada marca 'YYYYMMDDHH'."""
    return np.array([int(str(t)[-2:]) for t in times])


def corte_temporal(times, inicio_val=INICIO_VAL):
    """Máscaras booleanas (train, val): val son las marcas desde `inicio_val` en adelante."""
    val = np.asarray(times).astype(str) >= inicio_val
    trn = ~val
    if not trn.any() or not val.any():
        raise ValueError(f"El corte {inicio_val} deja vacío el entrenamiento o la validación.")
    return trn, val


def climatologia(Y, times):
    """Media por hora UTC: (24, filas, columnas, variables). Exige datos en las 24 horas."""
    h = horas(times)
    faltan = sorted(set(range(24)) - set(h.tolist()))
    if faltan:
        raise ValueError(f"No hay datos para las horas UTC {faltan}.")
    return np.stack([Y[h == k].mean(0) for k in range(24)])


def rmse_por_variable(pred, real):
    """RMSE de cada variable sobre todas las horas y celdas. Entradas (N, filas, columnas, 4)."""
    if pred.shape != real.shape:
        raise ValueError(f"Formas distintas: pred {pred.shape} y real {real.shape}.")
    rmse = np.sqrt(((pred - real) ** 2).mean(axis=(0, 1, 2)))
    if not np.isfinite(rmse).all():
        raise ValueError("Hay NaN o infinitos en la predicción o en el valor real.")
    return rmse


def score(rmse, escalas):
    """1 - media de rmse/escala; las cuatro variables pesan igual."""
    return float(1 - (rmse / escalas).mean())


def escala_climatologia(Y, times, inicio_val=INICIO_VAL):
    """RMSE en validación de la climatología calculada solo con train (denominador de score_local)."""
    t = np.asarray(times).astype(str)
    trn, val = corte_temporal(t, inicio_val)
    clim = climatologia(Y[trn], t[trn])
    return rmse_por_variable(clim[horas(t[val])], Y[val])


def validar(pred_val, Y, times, inicio_val=INICIO_VAL):
    """
    Evalúa una predicción de la validación.

    pred_val : (N_val, 70, 35, 4) en unidades físicas, en el orden de las horas de validación.
    Y, times : target y marcas de TODO train.npz (de ahí salen el corte y la climatología).
    """
    t = np.asarray(times).astype(str)
    _, val = corte_temporal(t, inicio_val)
    real, t_val = Y[val], t[val]
    rmse = rmse_por_variable(pred_val, real)
    evaluadas = np.isin(horas(t_val), HORAS_EVALUADAS)
    rmse_3h = rmse_por_variable(pred_val[evaluadas], real[evaluadas])
    return {
        "score_local": score(rmse, escala_climatologia(Y, t, inicio_val)),
        "rmse": {v: float(r) for v, r in zip(VARS, rmse)},
        "score_oficial": score(rmse_3h, ESCALAS_OFICIALES),
        "rmse_horas_evaluadas": {v: float(r) for v, r in zip(VARS, rmse_3h)},
        "horas_val": int(val.sum()),
        "horas_evaluadas": int(evaluadas.sum()),
        "primera": str(min(t_val)),
        "ultima": str(max(t_val)),
    }

In [ ]:
# COPIA EXACTA de train/ventanas.py: no editar aquí
"""
Ventanas (t-2, t-1, t) del GFS sin cruzar huecos de horas (T11 · HU-09). Solo NumPy.

La data del concurso tiene huecos de horas y de días. Una ventana está completa si existen las k horas
consecutivas que terminan en t. En las incompletas, la hora que falta se rellena con la hora t (igual que
el starter): sirven para predecir horas que Kaggle exige, pero se excluyen del entrenamiento.

Las horas previas se buscan por fecha, no por posición: nunca entra una hora del otro lado de un hueco.

El notebook lleva una copia exacta de este archivo en una celda para correr solo en Kaggle;
tests/test_ventanas.py comprueba que ambas sean iguales.
"""
from datetime import datetime, timedelta

import numpy as np


def indices_ventanas(times, k=3):
    """
    Índices de las horas (t-(k-1), …, t) de cada marca 'YYYYMMDDHH'.

    Devuelve (idx, completa): idx es (N, k), con el índice de t donde falta una hora previa;
    completa es una máscara (N,), True si las k horas existen.
    """
    fechas = [datetime.strptime(str(t), "%Y%m%d%H") for t in times]
    pos = {f: i for i, f in enumerate(fechas)}
    if len(pos) != len(fechas):
        raise ValueError("Hay horas repetidas en times.")
    idx = np.empty((len(fechas), k), np.int64)
    completa = np.ones(len(fechas), bool)
    for i, f in enumerate(fechas):
        for j, atras in enumerate(range(k - 1, -1, -1)):
            previa = pos.get(f - timedelta(hours=atras))
            if previa is None:
                completa[i] = False
                previa = i
            idx[i, j] = previa
    return idx, completa


def armar_ventanas(X, times, k=3):
    """Ventanas (N, k, ...) en float32 y la máscara de las completas."""
    if len(X) != len(times):
        raise ValueError(f"X tiene {len(X)} horas y times {len(times)}.")
    idx, completa = indices_ventanas(times, k)
    return np.asarray(X, np.float32)[idx], completa

In [ ]:
# La hora que falta en una ventana incompleta se rellena con la hora t; esas ventanas no entrenan (T11)
import functools

@functools.lru_cache(maxsize=None)
def ventanas_de(k):
    WTR, completa = armar_ventanas(Xtr, ttr, k)
    return WTR, completa

print('ventanas completas (3 h):', int(ventanas_de(3)[1].sum()), 'de', len(ttr))

In [ ]:
import json, platform
assert QUICK or torch.cuda.is_available(), 'Activa la GPU (Settings → Accelerator → GPU T4)'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
EQUIPO = torch.cuda.get_device_name(0) if dev == 'cuda' else 'CPU'
print(EQUIPO, '·', platform.python_version(), np.__version__, torch.__version__)
# Dos cortes temporales: A (el de T02) y B (1-15 jun, sin mirar nada posterior)
VENTANAS = {'A': dict(usar=np.ones(len(ttr), bool), ini='2024061600'),
            'B': dict(usar=ttr < '2024061600', ini='2024060100')}
for k, v in VENTANAS.items():
    v['trn'] = v['usar'] & (ttr < v['ini']); v['val'] = v['usar'] & (ttr >= v['ini'])
    print(f"ventana {k}: train {v['trn'].sum()} h · val {v['val'].sum()} h · {ttr[v['val']][0]} → {ttr[v['val']][-1]}")
def to_t(a): return torch.tensor(np.asarray(a), dtype=torch.float32, device=dev)
ESC = to_t(ESCALAS_OFICIALES)

In [ ]:
# Grilla WRF dentro de la grilla GFS (como en el starter): GFS fila 0 = norte, WRF y DEM fila 0 = sur
glat, glon = st['gfs_lats'], st['gfs_lons']
wlat, wlon = st['wrf_lats'], st['wrf_lons']
LAT, LON = np.meshgrid(wlat, wlon, indexing='ij')
gy = (LAT - glat[0]) / (glat[-1] - glat[0]) * 2 - 1
gx = (LON - glon[0]) / (glon[-1] - glon[0]) * 2 - 1
GRID = to_t(np.stack([gx, gy], -1)[None])                      # (1,70,35,2)
DEM = to_t((dem - dem.mean()) / dem.std())[None, None]

def media_local(a, k):
    """Media en una ventana k x k con borde reflejado."""
    p = np.pad(a, k // 2, mode='reflect')
    return np.lib.stride_tricks.sliding_window_view(p, (k, k)).mean((-1, -2))

def relieve(dem):
    """Pendiente (norte, este, magnitud), orientación (sin, cos) y relieve relativo (~15 y ~45 km)."""
    z = dem.astype(np.float64)
    dn, de = np.gradient(z)                                    # fila crece hacia el norte, columna al este
    mag = np.hypot(dn, de)
    unit = np.where(mag > 0, mag, 1.0)
    capas = [dn, de, mag, de / unit, dn / unit, z - media_local(z, 5), z - media_local(z, 15)]
    return np.stack([(c - c.mean()) / c.std() for c in capas])

TERRENO = to_t(relieve(dem))[None]                             # (1,7,70,35)

def interpolar(a):
    """(B,C,17,13) GFS -> (B,C,70,35) WRF, bilineal."""
    return F.grid_sample(a, GRID.expand(len(a), -1, -1, -1), align_corners=True)

class Bloque(nn.Module):
    def __init__(self, w, d, p=0.0):
        super().__init__()
        self.p = p
        self.f = nn.Sequential(nn.Conv2d(w, w, 3, padding=d, dilation=d), nn.GELU(),
                               nn.Conv2d(w, w, 3, padding=d, dilation=d))
    def forward(self, x):
        return F.gelu(x + F.dropout2d(self.f(x), self.p, self.training))

class Net(nn.Module):
    """Red de la ronda 2. `nv` = número de variables de entrada por hora."""
    def __init__(self, skip=False, terreno=False, profundo=False, w=64,
                 bloques=(1, 2, 4, 1), enc_prof=False, dropout=0.0, lags=3, nv=4):
        super().__init__()
        self.skip, self.terreno = skip, terreno
        self.enc = nn.Sequential(nn.Conv2d(nv * lags + 2, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, w, 3, padding=1), nn.GELU())
        self.static = nn.Parameter(torch.zeros(1, 8, 70, 35))
        self.enc2 = nn.Sequential(Bloque(w, 1, dropout), Bloque(w, 1, dropout)) if enc_prof else nn.Identity()
        c = w + 9 + (nv * lags if skip else 0) + (7 if terreno else 0)
        if profundo:
            self.dec = nn.Sequential(nn.Conv2d(c, w, 3, padding=1), nn.GELU(),
                                     *[Bloque(w, d, dropout) for d in bloques], nn.Conv2d(w, 4, 1))
        else:
            self.dec = nn.Sequential(nn.Conv2d(c, w, 3, padding=1), nn.GELU(),
                                     nn.Conv2d(w, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, 4, 1))
    def forward(self, x, h):                     # x: (B,lags,17,13,4), h: (B,) hora UTC
        B = x.shape[0]
        x = x.permute(0, 1, 4, 2, 3).reshape(B, -1, 17, 13)
        ang = 2 * np.pi * h / 24
        hh = torch.stack([torch.sin(ang), torch.cos(ang)], 1)[..., None, None].expand(B, 2, 17, 13)
        partes = [interpolar(self.enc2(self.enc(torch.cat([x, hh], 1)))), DEM.expand(B, -1, -1, -1),
                  self.static.expand(B, -1, -1, -1)]
        if self.skip:
            partes.append(interpolar(x))
        if self.terreno:
            partes.append(TERRENO.expand(B, -1, -1, -1))
        return self.dec(torch.cat(partes, 1)).permute(0, 2, 3, 1)  # (B,70,35,4) anomalía normalizada

In [ ]:
# ==== Código propio (solo si tu cambio es de entradas o de red) ====
def entradas_gfs(W):
    """W: (..., 4) GFS en unidades físicas (RH, T, U, V). Devuelve (..., nv). El residuo siempre usa las 4 originales."""
    return W

ENTRADAS = {'gfs': entradas_gfs}          # E1: añade aquí tu función y usa dict(entradas='nombre')
ARQUITECTURAS = {'base': Net}             # E3/E4: define tu clase (mismos argumentos que Net) y regístrala

In [ ]:
def datos_de(mascara, cfg):
    """Normalización y climatología SOLO con las horas de `mascara`."""
    t = ttr[mascara]
    E = np.asarray(ENTRADAS[cfg['entradas']](Xtr[mascara]), np.float32)
    mu_x = E.mean((0, 1, 2)); sd_x = E.std((0, 1, 2))
    assert (sd_x > 0).all(), 'Hay una variable de entrada constante'
    clim = climatologia(Ytr[mascara], t)
    anom_sd = (Ytr[mascara] - clim[horas(t)]).std((0, 1, 2))
    return {'cfg': cfg, 'CLIM': to_t(clim), 'ASD': to_t(anom_sd),
            'stats': dict(mu_x=mu_x, sd_x=sd_x, clim=clim, anom_sd=anom_sd)}

def preparar_x(D, W, t):
    """Entrada normalizada, hora y GFS interpolado como término base del residuo."""
    s = D['stats']
    x = to_t((np.asarray(ENTRADAS[D['cfg']['entradas']](W), np.float32) - s['mu_x']) / s['sd_x'])
    h = to_t(horas(t)).long()
    partes = []
    for i in range(0, len(W), 256):
        gfs = to_t(W[i:i + 256, -1])
        b = interpolar(gfs.permute(0, 3, 1, 2)).permute(0, 2, 3, 1) - D['CLIM'][h[i:i + 256]]
        partes.append(b / D['ASD'])
    return x, h, torch.cat(partes)

def red_de(cfg, nv):
    claves = ('skip', 'terreno', 'profundo', 'w', 'bloques', 'enc_prof', 'dropout', 'lags')
    return ARQUITECTURAS[cfg['arq']](nv=nv, **{k: cfg[k] for k in claves}).to(dev)

def entrenar(D, tensores, epocas, semilla):
    x, h, b, y = tensores
    cfg = D['cfg']; lote = cfg['lote']
    torch.manual_seed(semilla); torch.cuda.manual_seed_all(semilla)
    rng = np.random.default_rng(semilla)
    model = red_de(cfg, x.shape[-1])
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    n = len(x); pasos = epocas * int(np.ceil(n / lote))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg['lr'], total_steps=pasos)
    for ep in range(epocas):
        model.train()
        orden = torch.tensor(rng.permutation(n), device=dev)
        for s in range(0, n, lote):
            i = orden[s:s + lote]
            pred = model(x[i], h[i].float()) + b[i]
            mse = ((pred - y[i]) ** 2).mean(dim=(0, 1, 2)) * (D['ASD'] / ESC) ** 2
            loss = torch.sqrt(mse + 1e-8).mean()       # pérdida 'rmse' de T08
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return model

def tensores_de(D, mascara):
    W, completa = ventanas_de(D['cfg']['lags'])
    usar = np.where(mascara & completa)[0]
    if QUICK:
        usar = usar[:64]
    x, h, b = preparar_x(D, W[usar], ttr[usar])
    y = (to_t(Ytr[usar]) - D['CLIM'][h]) / D['ASD']
    return x, h, b, y

def predecir_lote(model, D, W, t, bs=128):
    x, h, b = preparar_x(D, W, t); model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(W), bs):
            s = slice(i, i + bs)
            p = model(x[s], h[s].float()) + b[s]
            out.append((D['CLIM'][h[s]] + p * D['ASD']).cpu().numpy())
    P = np.concatenate(out)
    P[..., 0] = np.clip(P[..., 0], 0, 100)
    return P

def medir(P, k):
    v = VENTANAS[k]
    r = validar(P, Ytr[v['usar']], ttr[v['usar']], inicio_val=v['ini'])
    return {'score_oficial': r['score_oficial'], 'score_local': r['score_local'], 'rmse_3h': r['rmse_horas_evaluadas']}

def linea(etiqueta, m):
    v = ' '.join(f'{k} {x:.3f}' for k, x in m['rmse_3h'].items())
    print(f"{etiqueta:34s} oficial {m['score_oficial']:.5f} · local {m['score_local']:.5f} · {v}", flush=True)

In [ ]:
SEMILLAS = [0, 1]
SALIDA = f'exp_{ID}'
os.makedirs(SALIDA, exist_ok=True)

def evaluar(nombre, cfg):
    """2 semillas en cada corte; guarda pesos, configuración y predicciones de validación."""
    ep = 1 if QUICK else cfg['epocas']
    por = {}
    for k, v in VENTANAS.items():
        D = datos_de(v['trn'], cfg); T = tensores_de(D, v['trn'])
        W = ventanas_de(cfg['lags'])[0][v['val']]
        por[k], preds = [], {}
        for semilla in SEMILLAS:
            t0 = time.time()
            mod = entrenar(D, T, ep, semilla)
            P = predecir_lote(mod, D, W, ttr[v['val']])
            m = medir(P, k)
            linea(f'{nombre} {k} s{semilla} ({time.time() - t0:.0f}s)', m)
            por[k].append(m); preds[f's{semilla}'] = P.astype(np.float32)
            torch.save(mod.state_dict(), f'{SALIDA}/{nombre}_{k}_s{semilla}.pt')
        np.savez(f'{SALIDA}/{nombre}_{k}_val.npz', times=ttr[v['val']], **preds)
        np.savez(f'{SALIDA}/{nombre}_{k}_estadisticas.npz', **D['stats'])
        del D, T, mod; torch.cuda.empty_cache()
    json.dump(cfg, open(f'{SALIDA}/{nombre}_config.json', 'w'))
    medias = {k: float(np.mean([m['score_oficial'] for m in r])) for k, r in por.items()}
    return {'nombre': nombre, 'cfg': cfg, 'media': float(np.mean(list(medias.values()))), 'por_ventana': medias,
            'rmse_3h': {x: float(np.mean([m['rmse_3h'][x] for r in por.values() for m in r])) for x in VARS},
            'todas': por}

assert 'base' not in CANDIDATOS
for nombre, cambio in CANDIDATOS.items():
    assert len(cambio) == 1, f'{nombre}: un solo cambio por candidato'
    assert set(cambio) <= set(BASE), f'{nombre}: clave desconocida {set(cambio) - set(BASE)}'
    assert any(BASE[c] != x for c, x in cambio.items()), f'{nombre}: es igual a la base'
base = evaluar('base', dict(BASE))
registro = [base]
for nombre, cambio in CANDIDATOS.items():
    r = evaluar(nombre, {**BASE, **cambio})
    r['delta'] = r['media'] - base['media']
    r['delta_ventana'] = {k: r['por_ventana'][k] - base['por_ventana'][k] for k in VENTANAS}
    r['aceptado'] = bool(r['delta'] >= 0.001 and all(d > 0 for d in r['delta_ventana'].values()))
    registro.append(r)
json.dump({'id': ID, 'autor': AUTOR, 'hipotesis': HIPOTESIS, 'quick': QUICK, 'resultados': registro,
           'versiones': {'python': platform.python_version(), 'numpy': np.__version__, 'torch': torch.__version__, 'equipo': EQUIPO}},
          open(f'{SALIDA}/resultados.json', 'w'), indent=1)

In [ ]:
# Fila lista para pegar en docs/modelo/experimentos.md (completa fecha, kernel y versión)
def resumen(r):
    v = ', '.join(f'{x:.3f}' for x in r['rmse_3h'].values())
    return f"{r['nombre']} {r['media']:.5f} (A {r['por_ventana']['A']:.5f}, B {r['por_ventana']['B']:.5f}; RH, T, U, V: {v})"

val = ' · '.join(resumen(r) + (f" {r['delta']:+.5f}" if 'delta' in r else '') for r in registro)
dec = ' · '.join(f"{r['nombre']}: {'se adopta' if r['aceptado'] else 'se descarta'} "
                 f"({', '.join(f'{k} {d:+.5f}' for k, d in r['delta_ventana'].items())})" for r in registro[1:])
aviso = 'PRUEBA CORTA, NO REGISTRAR · ' if QUICK else ''
print(f"| <fecha> | {AUTOR} | {aviso}**{ID}** ({HIPOTESIS}). `train/plantilla_experimento.ipynb`, Kaggle `<usuario/kernel>` v<n>, "
      f"{EQUIPO}, torch {torch.__version__}, semillas 0 y 1, cortes A y B | {val} | no enviado | {dec} |")